# Customer Service Email Classification Agent

This notebook implements a DSPy-based email classifier for customer service tickets.
The agent classifies incoming emails into predefined contact reasons based on the subject and first message.

## 0. Imports

In [1]:
import dspy
import mlflow
import pandas as pd
from typing import Literal
import json
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

c:\code\customer-service-agent\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. LM Configuration

Configure the Azure OpenAI language model using DefaultAzureCredential for authentication.
The model follows the LiteLLM provider format: `azure/<deployment-name>`.

In [3]:
# === MLflow Tracing Setup ===
mlflow.dspy.autolog()
mlflow.set_experiment("dspy-email-classifier")

# === Load Environment ===
load_dotenv()

credential = DefaultAzureCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default").token

# === Configure Language Models ===
# DSPy uses LiteLLM under the hood - for Azure OpenAI use format: azure/<deployment-name>
lm_1 = dspy.LM(
    model="azure/gpt-4.1",
    api_base="https://ai-ecom-data-agent-resource.cognitiveservices.azure.com",
    api_key=token
)

lm_2 = dspy.LM(
    model="azure/gpt-4.1-mini",
    api_base="https://ai-ecom-data-agent-resource.cognitiveservices.azure.com",
    api_key=token,
    cache=False
)

# Set default LM for DSPy modules
dspy.configure(lm=lm_2)

print(f"✓ MLflow tracing enabled")
print(f"✓ LM configured: {lm_1.model}")
print(f"✓ LM configured: {lm_2.model}")

# Test the connection (will be traced!)
response = lm_1("Say 'Hello!' if you can hear me.")
response = lm_2("Say 'Hello!' if you can hear me.")

print(f"✓ Connection test: {response[:50]}...")

✓ MLflow tracing enabled
✓ LM configured: azure/gpt-4.1
✓ LM configured: azure/gpt-4.1-mini
✓ Connection test: ['Hello!']...


In [4]:
lm_2("hello, which model provider is the best?")

['Hello! The "best" model provider really depends on your specific needs, such as the type of task, budget, required language support, deployment preferences, and other factors. Here are a few well-known model providers as of 2024:\n\n1. **OpenAI**  \n   - Strengths: State-of-the-art language models like GPT-4, strong support and ecosystem, widely used APIs.  \n   - Good for: Advanced conversational AI, content generation, code completion, research, and enterprise solutions.\n\n2. **Google (Vertex AI)**  \n   - Strengths: Integration with Google Cloud Platform, strong in multi-modal AI (language, vision), reliable infrastructure.  \n   - Good for: Enterprises already using Google Cloud, large-scale AI projects.\n\n3. **Anthropic**  \n   - Strengths: Focus on AI safety and alignment, competitive large language models like Claude.  \n   - Good for: Applications emphasizing safety and ethical AI use.\n\n4. **Cohere**  \n   - Strengths: Custom embedding and generation models with easy-to-u

In [5]:
lm_1("hello, which model provider is the best?")

['Hello! The "best" model provider depends on what you need—there isn’t a single answer for everyone. Here’s a quick overview of major AI model providers and their strengths as of 2024:\n\n### 1. **OpenAI (makers of ChatGPT, GPT-4 and GPT-4o)**\n   - **Strengths:** State-of-the-art language models, high accuracy, conversational depth, great tooling (APIs, plugins), reliable ethics guardrails.\n   - **Best for:** General-purpose chatbots, creative writing, coding help, research, and enterprise applications.\n\n### 2. **Google (Gemini, formerly Bard)**\n   - **Strengths:** Integrates well with Google ecosystem, powerful with factual retrieval, strong in reasoning tasks, leading-edge research.\n   - **Best for:** Web search integration, summarization, and real-time information tasks.\n\n### 3. **Anthropic (Claude 3 series)**\n   - **Strengths:** Advanced safety, very long context windows (can process large documents), transparent model behavior.\n   - **Best for:** Business/enterprise, do

In [6]:
with open("labels.json", encoding="utf-8") as f:
    config = json.load(f)

LABELS = config["labels"]
CONTACT_REASONS = list(LABELS.keys())

print(f"✓ {len(CONTACT_REASONS)} contact reasons loaded:")
for reason in CONTACT_REASONS:
    print(f"  - {reason}: {LABELS[reason]['description'][:50]}...")

✓ 12 contact reasons loaded:
  - Order Delay: Klant vraagt naar verzendstatus, track & trace upd...
  - Lost Order: Pakket staat als afgeleverd maar klant geeft aan h...
  - Return Order: Klant wil een product retourneren voor terugbetali...
  - Cancel Order: Klant wil een bestelling annuleren voordat deze is...
  - Damaged Item: Product is kapot, gebarsten, gescheurd of beschadi...
  - Bad Product Quality: Product is defect, werkt niet zoals verwacht, of k...
  - Wrong Order: Klant heeft verkeerd product ontvangen, verkeerde ...
  - Missing Item: Bestelling is aangekomen maar één of meerdere arti...
  - Product Question: Vragen over productspecificaties, eigenschappen, c...
  - Shipping Question: Algemene vragen over verzendopties, kosten, levert...
  - Special Request: Aangepaste verzoeken zoals cadeauverpakking, speci...
  - Other: Algemene vragen die niet in andere categorieën pas...


In [ ]:
# def build_label_descriptions() -> str:
#     """Build label descriptions for the signature."""
#     return "\n".join([
#         f"- {key}: {info['description']}" 
#         for key, info in LABELS.items()
#     ])

# # Preview the label descriptions
# print(build_label_descriptions())

In [ ]:
def get_label_descriptions(label:str) -> str:
    """Get label description for a specific label."""
    return f"{label}: {LABELS[label]['description']}"

# Preview the label descriptions
print(get_label_descriptions("Order Delay"))

Order Delay: Klant vraagt naar verzendstatus, track & trace updates, of vertraagde levering. Bestelling is niet binnen de verwachte termijn aangekomen.


## 3. Signature Definition

Define a DSPy Signature for the classification task. The signature specifies:
- **Inputs**: `subject` and `first_message` from the customer email
- **Output**: `contact_reason` - one of the predefined categories (using `Literal` type)

In [43]:
# Contact signature class met inheritance van dspy.Signature class
class LLMJudgeSignature(dspy.Signature):
    """Judge the most appropriate contact reason for a customer email."""
    
    subject: str = dspy.InputField(desc="The email subject line")
    first_message: str = dspy.InputField(desc="The first customer message")
    label_a: str = dspy.InputField(desc="Label option A")
    label_b: str = dspy.InputField(desc="Label option B")
    label_a_definition: str = dspy.InputField(desc="Definition of label A")
    label_b_definition: str = dspy.InputField(desc="Definition of label B")    
    
    best_label: Literal["A", "B"] = dspy.OutputField(
        desc="Choose 'A' for label_a or 'B' for label_b"
    )

## 4. Classifier Module

Create a DSPy Module that wraps the signature with a predictor. The `forward` method defines how inputs flow through the module.

In [44]:
class LLMJudgeClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predictor = dspy.Predict(LLMJudgeSignature)

    def forward(self, subject: str, first_message: str, label_a: str, label_b: str) -> str:
        result = self.predictor(subject=subject, first_message=first_message, 
                            label_a=label_a, label_b=label_b,
                            label_a_definition=get_label_descriptions(label_a),
                            label_b_definition=get_label_descriptions(label_b)
                            ).best_label
        return label_a if result == "A" else label_b

# Initialize the classifier
classifier = LLMJudgeClassifier()
print("✓ LLMJudgeClassifier module initialized")

✓ LLMJudgeClassifier module initialized


## 5. Dataset Preparation

Utility function to convert a pandas DataFrame into DSPy Examples for training/evaluation.

In [45]:
def prepare_dataset(df: pd.DataFrame):
    """Convert DataFrame to DSPy Examples."""
    dataset = []
    for _, row in df.iterrows():
        example = dspy.Example(
            subject=row['subject'] or "",
            first_message=row['first_message'] or "",
            label_a=row['actual_label'] or "",
            label_b=row['predicted_label'] or ""
        ).with_inputs('subject', 'first_message', 'label_a', 'label_b')
        dataset.append(example)
    return dataset

print("✓ prepare_dataset function defined")

✓ prepare_dataset function defined


## 6. Load Dataset

Load the cleaned ticket dataset and prepare it for evaluation.

In [46]:
# Load the dataset
df = pd.read_parquet("../data/classification_results.parquet")
print(f"✓ {len(df)} tickets loaded")

df_discrepancies = df[df['potential_mislabel'] == True]

# Display sample
df_discrepancies.count()

✓ 333 tickets loaded


subject               73
first_message         73
actual_label          73
predicted_label       73
match                 73
potential_mislabel    73
dtype: int64

In [48]:
# Prepare the dataset for DSPy
df_prepared = prepare_dataset(df_discrepancies)
print(f"✓ {len(df_prepared)} examples prepared for DSPy")

✓ 73 examples prepared for DSPy


In [49]:
pred = classifier(
    subject=df_prepared[2].subject,
    first_message=df_prepared[2].first_message,
    label_a=df_prepared[2].label_a,
    label_b=df_prepared[2].label_b
)

print(f"Predicted contact reason: {pred}, label_a: {df_prepared[2].label_a}, label_b: {df_prepared[2].label_b}")

2026/01/31 10:31:49 WARNING dspy.primitives.module: Failed to set LM usage. Please return `dspy.Prediction` object from dspy.Module to enable usage tracking.


Predicted contact reason: Other, label_a: Order Delay, label_b: Other


In [33]:
import polars as pl

df_pl = pl.read_parquet("../data/classification_results.parquet")

df_pl = df_pl.sql("SELECT * FROM self WHERE potential_mislabel = true")

In [50]:
from dspy.evaluate import Evaluate

# Define metric for classification
def classification_metric(example, pred, trace=False):
    """Returns 1 if prediction matches label_b, 0 otherwise."""
    return example.label_b == pred

# Set up the evaluator (DSPy best practice)
evaluator = Evaluate(
    devset=df_prepared,
    metric=classification_metric,
    num_threads=4,  # Parallel evaluation
    display_progress=True,
    display_table=10  # Show first 10 results in table
)

# Run evaluation
eval_result = evaluator(classifier)

print(f"\n✓ Evaluation complete!")
print(f"  Accuracy: {eval_result.score:.1f}%")

  0%|          | 0/73 [00:00<?, ?it/s]

2026/01/31 10:31:59 WARNING dspy.primitives.module: Failed to set LM usage. Please return `dspy.Prediction` object from dspy.Module to enable usage tracking.


Average Metric: 1.00 / 1 (100.0%):   1%|▏         | 1/73 [00:01<01:32,  1.29s/it]

2026/01/31 10:31:59 WARNING dspy.primitives.module: Failed to set LM usage. Please return `dspy.Prediction` object from dspy.Module to enable usage tracking.


Average Metric: 2.00 / 2 (100.0%):   3%|▎         | 2/73 [00:01<00:45,  1.57it/s]

...

2026/01/31 10:32:17 WARNING dspy.primitives.module: Failed to set LM usage. Please return `dspy.Prediction` object from dspy.Module to enable usage tracking.
2026/01/31 10:32:17 WARNING dspy.primitives.module: Failed to set LM usage. Please return `dspy.Prediction` object from dspy.Module to enable usage tracking.


Average Metric: 60.00 / 73 (82.2%): 100%|██████████| 73/73 [00:19<00:00,  3.73it/s]

2026/01/31 10:32:17 INFO dspy.evaluate.evaluate: Average Metric: 60 / 73 (82.2%)


,subject,first_message,label_a,label_b,prediction,_patched
0,38556826: Re: Jouw bestelling is verzonden!,"Goedemorgen, Meer dan een week geleden is de deurmat die we bij ju...",Special Request,Order Delay,Order Delay,✔️ [True]
1,38473261: Klacht,"Geachte heer/mevrouw, Onlangs heb ik bij u een bestelling geplaats...",Order Delay,Bad Product Quality,Bad Product Quality,✔️ [True]
2,38454445: Bestelling hondentuig,"Goedemiddag, Ik heb twee tuigjes besteld voor onze honden Tommie e...",Order Delay,Other,Other,✔️ [True]
3,38438330: Nieuw klantbericht op 23 januari 2026 om 10:40,Nieuw klantbericht op 23 januari 2026 om 10:40 Je hebt een nieuw b...,Return Order,Wrong Order,Return Order,✔️ [False]
4,38251988:,Hoe lang duurt de verzending???,Order Delay,Shipping Question,Shipping Question,✔️ [True]
5,38194422: Re: Je bestelling met nummer #18946 is onderweg!,"Hallo , ik heb deze al ontvangen Mvg De Winne Dorine Van: Info | M...",Order Delay,Other,Other,✔️ [True]
6,38137964: Waar blijft mijn bestelling van vorig jaar?,Goedenavond \n\nHeb tot op heden niets vernomen van mijn bestellin...,Order Delay,Lost Order,Order Delay,✔️ [False]
7,38088109: Nieuw klantbericht op 21 januari 2026 om 15:57,Nieuw klantbericht op 21 januari 2026 om 15:57 Je hebt een nieuw b...,Order Delay,Special Request,Special Request,✔️ [True]
8,38018545: Klacht Bestelling #19178,"Beste, Vorige week heb ik eindelijk de 2 speelballen binnen gekreg...",Damaged Item,Bad Product Quality,Bad Product Quality,✔️ [True]
9,37926145: Re: Bestelling #19622 bevestigd,"Beste, De deurmat is vandaag aangekomen, maar het Padded Anti Trek...",Order Delay,Missing Item,Missing Item,✔️ [True]



✓ Evaluation complete!
  Accuracy: 82.2%


In [56]:

# Gold dataset evt aanmaken met menselijke controle om te controleren waar het model slecht in was, waar _patched false is 

# eval_result heeft een results attribuut met alle voorspellingen
# Methode 1: Direct via de results
results_df = pd.DataFrame([
    {
        "subject": ex.subject,
        "first_message": ex.first_message,  # truncate
        "label_a": ex.label_a,
        "label_b": ex.label_b,
        "prediction": pred,
        "ai_is_right": ex.label_b == pred,
        "score": score
    }
    for ex, pred, score in eval_result.results
])

# Methode 2: Bekijk eerst de structuur
print(dir(eval_result))
print(eval_result.results[:2])  # Bekijk eerste 2 items


['__add__', '__class__', '__contains__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__float__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__radd__', '__reduce__', '__reduce_ex__', '__repr__', '__rtruediv__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', '__truediv__', '__weakref__', '_completions', '_lm_usage', '_store', 'completions', 'copy', 'from_completions', 'get', 'get_lm_usage', 'inputs', 'items', 'keys', 'labels', 'set_lm_usage', 'toDict', 'values', 'with_inputs', 'without']
[(Example({'subject': '38556826: Re: Jouw bestelling is verzonden!', 'first_message': 'Goedemorgen, \n\nMeer dan een week geleden is de deurmat die we bij jullie hebben besteld op de post gegaan, helaas hebben we nog niks ontvangen. \nWanneer mogen we het verwachten? \n\nMvg, \n\nJoan

## 8. Inspect LM History

Use `dspy.inspect_history()` to see the prompts sent to the LM and the responses received.

In [ ]:
# Inspect the last LM call to see the prompt and response
lm.inspect_history(n=1)

In [ ]:
"""
Cell generated by Data Wrangler.

# Zelf classificeren
# Reasioning van het model onderzoeken
# Golden dataset maken, en als few shot example meegeven in een volgende itteratie

"""
def clean_data(results_df):
    # Filter rows based on column: 'ai_is_right'
    results_df = results_df[results_df['ai_is_right'].apply(str).str.contains("False", regex=False, na=False, case=False)]
    return results_df

results_df_clean = clean_data(results_df.copy())
results_df_clean.head()

,subject,first_message,label_a,label_b,prediction,ai_is_right,score
3,38438330: Nieuw klantbericht op 23 januari 2026 om 10:40,Nieuw klantbericht op 23 januari 2026 om 10:40 \n \n\nJe hebt ee...,Return Order,Wrong Order,Return Order,False,False
6,38137964: Waar blijft mijn bestelling van vorig jaar?,Goedenavond \n\nHeb tot op heden niets vernomen van mijn bestellin...,Order Delay,Lost Order,Order Delay,False,False
10,37903608: Update van uw bestelling - #19880,"Beste Ronny,\n\nHartelijk dank voor je bestelling bij Mivero! We n...",Order Delay,Wrong Order,Order Delay,False,False
13,37858208: Pakket niet ontvangen,"Goedendag, \n7 januari heb ik een bestelling bij jullie gedaan. He...",Order Delay,Lost Order,Order Delay,False,False
16,37729987: Nieuw klantbericht op 19 januari 2026 om 18:23,Nieuw klantbericht op 19 januari 2026 om 18:23 \n \n\nJe hebt ee...,Order Delay,Cancel Order,Order Delay,False,False


In [ ]:
# Controle welke mederwerk het vaakst het verkeerde label gebruikt
# Draaien op de hele dataset
# Accurancy blijven meten 
# View shot toevoegen, optimalisaties bekijken binnen DSPY
# Architectuur inrichting 
# - 1 Trigger of scheduled? 
# - 2 Batches of per bericht
# - 3 Lokaal model of cloud?
# - 4 Hosting mcp + model code
# - 5 CI/CD proces voor model + code
# - 6 Class dspy module voor de workflow classificatie vs bericht sturen (ondezoeken ReAct, Chain of Thought, Predict)
# - 7 Logging en monitoring (MLflow)
# - 8 Order delay pipeline definieren en maken (Class met methods voor voorspelbare uitkomsten)

In [ ]:
# === BootstrapFewShot Optimizer ===
# Verbeter de accurancy door automatisch few-shot voorbeelden te bootstrappen

from dspy.teleprompt import BootstrapFewShot

# Split dataset in train/test voor optimalisatie
train_size = int(len(df_prepared) * 0.7)
trainset = df_prepared[:train_size]
testset = df_prepared[train_size:]

print(f"✓ Dataset split: {len(trainset)} train, {len(testset)} test")

# Metric voor de optimizer (moet True/False returnen)
def bootstrap_metric(example, pred, trace=None):
    """Returns True if prediction matches the expected label."""
    # pred is de return waarde van classifier.forward() - dus het gekozen label
    return pred == example.label_b  # We verwachten dat AI label_b kiest (de AI predictie)

# 1. Maak de optimizer
optimizer = BootstrapFewShot(
    metric=bootstrap_metric,
    max_bootstrapped_demos=4,  # Genereer nieuwe voorbeelden via bootstrapping
    max_labeled_demos=8,       # Voeg voorbeelden toe uit je dataset
    max_rounds=1               # Aantal bootstrap rondes
)

print("✓ BootstrapFewShot optimizer geconfigureerd")

# 2. Compileer je programma met de trainset
print("⏳ Optimalisatie gestart...")
optimized_classifier = optimizer.compile(
    student=LLMJudgeClassifier(),  # Nieuwe instance van je classifier
    trainset=trainset
)

print("✓ Optimalisatie voltooid!")

# 3. Evalueer de geoptimaliseerde classifier op de testset
optimized_evaluator = Evaluate(
    devset=testset,
    metric=classification_metric,
    num_threads=4,
    display_progress=True,
    display_table=5
)

# Vergelijk baseline vs optimized
print("\n📊 Baseline classifier evaluatie:")
baseline_result = optimized_evaluator(classifier)
print(f"  Baseline Accuracy: {baseline_result.score:.1f}%")

print("\n📊 Geoptimaliseerde classifier evaluatie:")
optimized_result = optimized_evaluator(optimized_classifier)
print(f"  Optimized Accuracy: {optimized_result.score:.1f}%")

print(f"\n✓ Verbetering: {optimized_result.score - baseline_result.score:+.1f}%")